In [ ]:
# ✅ Step 1: Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.decomposition import PCA
import joblib
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Step  Upload Files (for Google Colab)
from google.colab import files
uploaded = files.upload()

In [ ]:
# ✅ Step 2: Load Dataset
main_df = pd.read_excel("LFB_2019_22.xlsx")
meta_df = pd.read_excel("LFB Metadata.xlsx")

# ✅ Step 3: Filter for a specific borough
filtered_df = main_df[main_df['IncGeo_BoroughCode'] == 'E09000008'].copy()
print(f"✅ Filtered rows: {filtered_df.shape[0]}")

# ✅ Step 4: Drop columns with >50% missing values
filtered_df = filtered_df.loc[:, filtered_df.isnull().mean() < 0.5]

# ✅ Step 5: Drop irrelevant or high-cardinality columns
drop_cols = ['IncidentNumber', 'Postcode_full', 'UPRN', 'USRN', 'ProperCase']
filtered_df.drop(columns=[col for col in drop_cols if col in filtered_df.columns], inplace=True)

# ✅ Step 6: Drop datetime columns
datetime_cols = filtered_df.select_dtypes(include='datetime').columns.tolist()
filtered_df.drop(columns=datetime_cols, inplace=True)
# ✅ Step 7: Fill missing values
for col in filtered_df.select_dtypes(include='number').columns:
    filtered_df[col] = filtered_df[col].fillna(filtered_df[col].median())
for col in filtered_df.select_dtypes(include='object').columns:
    filtered_df[col] = filtered_df[col].fillna(filtered_df[col].mode()[0])

# ✅ Step 8: Encode categorical features
le = LabelEncoder()
for col in filtered_df.select_dtypes(include='object').columns:
    filtered_df[col] = le.fit_transform(filtered_df[col])

In [ ]:
# ✅ Step: Enhanced Data Understanding
print("\n✅ Summary Statistics with Skewness & Kurtosis:")
desc_stats = filtered_df.describe().T
desc_stats["skewness"] = filtered_df.skew()
desc_stats["kurtosis"] = filtered_df.kurtosis()
print(desc_stats)

In [ ]:
for col in ['PumpHoursRoundUp', 'Notional Cost (£)', 'PumpCount', 'NumCalls']:
    filtered_df[col] = np.log1p(filtered_df[col])  # log1p to avoid log(0)


In [ ]:
print("\n✅ Value Ranges:")
for col in filtered_df.select_dtypes(include='number').columns:
    print(f"{col}: Min={filtered_df[col].min()}, Max={filtered_df[col].max()}")

In [ ]:
print("\n✅ Outlier Detection using IQR:")
for col in filtered_df.select_dtypes(include='number').columns:
    Q1 = filtered_df[col].quantile(0.25)
    Q3 = filtered_df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = filtered_df[(filtered_df[col] < Q1 - 1.5 * IQR) | (filtered_df[col] > Q3 + 1.5 * IQR)]
    print(f"{col}: {len(outliers)} outliers")
    # Boxplot example
plt.figure(figsize=(10, 4))
sns.boxplot(data=filtered_df[['FirstPumpArriving_AttendanceTime', 'Notional Cost (£)']])
plt.title("Boxplot of Selected Continuous Variables")
plt.show()


In [ ]:
# ✅ Step 9: Normalize data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(filtered_df)

# ✅ Step 10: Apply PCA
pca = PCA(n_components=0.95, random_state=42)
reduced_data = pca.fit_transform(scaled_data)

In [ ]:
# ✅ Step 11: Find Optimal K using Silhouette Score
silhouette_scores = []
K = range(2, 21)
for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
    score = silhouette_score(reduced_data, km.fit_predict(reduced_data))
    silhouette_scores.append(score)

In [ ]:
# ✅ Step 12: KMeans with Best K
best_k = K[np.argmax(silhouette_scores)]
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20, max_iter=500)
clusters = kmeans.fit_predict(reduced_data)
filtered_df['Cluster'] = clusters
print(f"✅ Best k: {best_k}")

# ✅ Step 12.5: Cluster Analysis
print("\n✅ Cluster Descriptions (Mean of Features per Cluster):")
cluster_summary = filtered_df.groupby('Cluster').mean()
print(cluster_summary)

print("\n✅ Cluster Sizes:")
print(filtered_df['Cluster'].value_counts())

# ✅ Boxplots to visualize distribution by cluster
for col in filtered_df.columns[:5]:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x='Cluster', y=col, data=filtered_df)
    plt.title(f"{col} Distribution by Cluster")
    plt.show()

In [ ]:
# ✅ Step 13: Decision Tree Classification based on Clusters
X = filtered_df.drop('Cluster', axis=1)
y = filtered_df['Cluster']

param_grid = {'max_depth': [3, 5, 10], 'min_samples_split': [2, 5, 10]}
grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5)
grid.fit(X, y)
best_clf = grid.best_estimator_
print("✅ Best Params:", grid.best_params_)

# ✅ Step 14: Train-Test Split & Fit
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
best_clf.fit(X_train, y_train)
y_pred = best_clf.predict(X_test)

In [ ]:
# ✅ Step 15: Evaluation Metrics
print("✅ Evaluation Metrics:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.2f}")
print(f"Recall: {recall_score(y_test, y_pred, average='weighted'):.2f}")
print(f"F1 Score: {f1_score(y_test, y_pred, average='weighted'):.2f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ✅ Cross-Validation
cv_scores = cross_val_score(best_clf, X, y, cv=5, scoring='accuracy')
print("✅ Cross-Validation Accuracy:", cv_scores.mean())

In [ ]:
# ✅ Step 16: Visualizations
# Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix Heatmap')
plt.show()


In [ ]:
plt.figure(figsize=(14, 10))
corr_matrix = filtered_df.corr()
sns.heatmap(corr_matrix, annot=False, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
# PCA Cluster Visualization
plt.figure(figsize=(8, 6))
plt.scatter(reduced_data[:, 0], reduced_data[:, 1], c=clusters, cmap='viridis', alpha=0.7)
plt.title("KMeans Clustering (2D PCA Projection)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.grid(True)
plt.show()

In [ ]:
# Feature Importance
importances = best_clf.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices], y=X.columns[indices])
plt.title("Feature Importance from Decision Tree")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Step 17: Save Models
joblib.dump(best_clf, 'decision_tree_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(pca, 'pca_model.pkl')

# ✅ Step 18: Visualize Decision Tree
plt.figure(figsize=(20, 10))
plot_tree(best_clf, feature_names=X.columns, class_names=[str(i) for i in np.unique(y)], filled=True)
plt.title("Decision Tree Visualization")
plt.show()
